# ThoughtZero: Kaggle GPU server template

Copy this notebook, then set **Settings → Accelerator → GPU T4 x2** and **Internet on**.

Add Kaggle *secrets* (Add-ons → Secrets): `TYPESAFE_API_KEY` (only for Jev runs) and, if the repo is private, `GITHUB_TOKEN`.
Never paste a key into a cell. **Clear all outputs before sharing.**

Status: draft. fp16 correctness on a T4 is unverified (docs/gpu_setup.md).

In [ ]:
# --- Parameters ---------------------------------------------------------
MODEL = "google/gemma-4-E4B-it-qat-w4a16-ct"   # 4-bit QAT; the bf16 E4B (16 GB) does not fit a T4
TOKENIZER = "google/gemma-4-E4B-it"
REPO = "github.com/<org>/thoughtzero"          # TODO: set
BRANCH = "main"
SCRIPT = "scripts/smoke_test.py --mock"        # what to run once the server is up

In [ ]:
!pip install -q -U "vllm>=0.19.1"   # TODO: pin the exact version once verified on a T4

In [ ]:
import os, subprocess, time, urllib.request

env = dict(os.environ, CUDA_VISIBLE_DEVICES="0")   # GPU 1 stays free for the PRM
server = subprocess.Popen(
    ["vllm", "serve", MODEL, "--dtype", "float16", "--max-model-len", "8192",
     "--enable-prefix-caching", "--max-logprobs", "20",
     "--host", "127.0.0.1", "--port", "8000"],
    env=env, stdout=open("vllm.log", "w"), stderr=subprocess.STDOUT,
)
for _ in range(180):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=2)
        print("server up"); break
    except Exception:
        if server.poll() is not None:
            raise RuntimeError("vLLM exited; see vllm.log")
        time.sleep(5)
else:
    raise TimeoutError("server did not come up; see vllm.log")

In [ ]:
# Sanity check: one greedy completion. Look for garbage/NaN output (fp16 overflow).
import json, urllib.request
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(TOKENIZER)
prompt = tok.apply_chat_template([{"role": "user", "content": "What is 2+3? Answer in \\boxed{}."}],
                                 tokenize=False, add_generation_prompt=True, enable_thinking=False)
req = urllib.request.Request("http://127.0.0.1:8000/v1/completions",
    data=json.dumps({"model": MODEL, "prompt": prompt, "max_tokens": 64, "temperature": 0,
                     "n": 1, "logprobs": 5}).encode(),
    headers={"Content-Type": "application/json"})
print(json.load(urllib.request.urlopen(req))["choices"][0]["text"])

In [ ]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

def secret(name):
    try:
        return secrets.get_secret(name)
    except Exception:
        return ""

with open(".env", "w") as f:
    f.write(f"GEMMA_BASE_URL=http://127.0.0.1:8000/v1\nGEMMA_MODEL={MODEL}\n")
    f.write(f"TYPESAFE_API_KEY={secret('TYPESAFE_API_KEY')}\n")

In [ ]:
gh = secret("GITHUB_TOKEN")
url = f"https://{gh + '@' if gh else ''}{REPO}@{BRANCH}"
subprocess.run(["pip", "install", "-q", f"git+{url}"], check=True)   # never print `url`

In [ ]:
!git clone -q --depth 1 -b {BRANCH} https://{REPO} repo 2>/dev/null || true
!cd repo && cp ../.env . && python {SCRIPT}

In [ ]:
# Copy results to the notebook output, then stop the server.
!mkdir -p /kaggle/working/results && cp -r repo/results/* /kaggle/working/results/ 2>/dev/null || true
server.terminate()